# Classifier Fallback 및 과금

Claude Fable 5는 cybersecurity, biology, chemistry 같은 분야에서 고도화된 기능을 갖추고 있어 실제 오용 위험이 있습니다. 유용성을 높이는 바로 그 능력이 악의적인 행위자의 cyberattack이나 위험한 무기 제작을 도울 수도 있습니다. 이러한 이유로 Claude Fable 5에는 이 특정 분야들에 대한 safeguard가 적용되며, 모든 요청마다 자동화된 safety check가 실행됩니다. 이 검사는 다음을 포함한 분야의 요청을 차단합니다:

- **공격적 cybersecurity 기법** — exploit, malware 또는 공격 도구 제작
- **Biology 및 life sciences** — 실험실 방법 또는 분자 메커니즘
- **경쟁 AI model 개발** — frontier AI system 구축에 도움이 될 수 있는 요청
- **Model의 [summarized thinking](https://platform.claude.com/docs/en/build-with-claude/extended-thinking#summarized-thinking) 추출**

이러한 safeguard는 의도적으로 보수적입니다. 우선 견고성을 목표로 조정되었으므로, 무해한 기술 작업에서도 때때로 작동합니다. 다른 모든 분야에서 Fable의 Mythos 수준 능력을 더 빠르게 제공하기 위해, biology 및 cybersecurity와 관련된 모든 주제에서는 Opus 4.8로 fallback하도록 Fable 5를 출시합니다. 출시 후에도 Fable 5의 false-positive 비율을 계속 낮춰 나갈 것입니다.

**API 고객은 Claude Fable 5에서 Opus 4.8로의 fallback을 구성해야 합니다.** [내장 server-side fallback](https://platform.claude.com/docs/en/build-with-claude/handling-stop-reasons#server-side-fallback) 기능(native Claude API 및 Claude Platform on AWS에서 사용 가능)이나 Anthropic SDK helper를 기반으로 구현한 [client-side fallback](https://platform.claude.com/docs/en/build-with-claude/handling-stop-reasons#client-side-fallback) 로직을 사용하세요.

출력이 발생하기 전에 차단된 일부 요청에는 요금이 부과되며, fallback 요청으로 이어지는 cached prompt prefix에는 새로운 cache write 요금 대신 할인된 cache-read 요금이 부과됩니다. **Server-side fallback을 사용하지 않을 때** 이 할인 요금을 적용하려면 fallback credit token이 필요하며, SDK의 fallback middleware가 이를 대신 사용해 줍니다. [아래](#4-billing-changes)를 참고하세요.

## 이 가이드에서 다루는 내용

1. [Classifier 차단의 형태](#1-what-a-classifier-block-looks-like)
2. [Server-side fallback(권장)](#2-server-side-fallback-recommended)
3. [Streaming](#3-streaming)
4. [과금 변경 사항](#4-billing-changes)
5. [SDK를 사용한 client-side fallback](#5-client-side-fallback-with-the-sdk)
6. [흔한 anti-pattern](#6-common-anti-patterns)


In [ ]:
%%capture
%pip install -U "anthropic>=0.108.0"

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

PRIMARY_MODEL = "claude-fable-5"
FALLBACK_MODEL = "claude-opus-4-8"
SERVER_SIDE_FALLBACK_BETA = "server-side-fallback-2026-06-01"
FALLBACK_CREDIT_BETA = "fallback-credit-2026-06-01"

# Anthropic()은 환경 변수에서 ANTHROPIC_API_KEY를 읽습니다. 실제 예제를 실행하기 전에
# .env 파일(위에서 load됨)에 추가하거나 shell에서 export하세요.
if not os.environ.get("ANTHROPIC_API_KEY"):
    print(
        "ANTHROPIC_API_KEY is not set - add it to .env or export it."
    )


## 1. Classifier 차단의 형태

*Classifier 차단*은 요청이 safeguard를 위반하는 것으로 보일 때 API가 반환하는 결과입니다. API는 `stop_reason: "refusal"`과 category를 설명하는 `stop_details` 객체를 포함하여 `200`을 반환합니다:

```json
{
  "stop_reason": "refusal",
  "stop_details": {
    "type": "refusal",
    "category": "cyber",
    "explanation": "This request triggered restrictions on violative cyber content and was blocked under Anthropic's Usage Policy..."
  },
  "content": [...]
}
```

로직은 **`content`나 `stop_details`가 아니라 `stop_reason`에 따라 분기하세요**. `stop_details`는 정보 제공용이며 `null`일 수 있습니다. 이 경우 일반적인 refusal(아래 category 중 특정 하나에 해당하지 않음)로 처리해야 합니다.

존재하는 경우 `category`는 `"cyber"`, `"bio"`, `"frontier_llm"`, `"reasoning_extraction"` 중 하나입니다. 이를 통해 fallback 선택을 세밀하게 조정할 수 있습니다:

| category | 작동 대상 |
| --- | --- |
| `cyber` | 공격적 cybersecurity 콘텐츠(exploit, malware, 공격 도구) |
| `bio` | biology / life-sciences 콘텐츠(실험실 방법, 분자 메커니즘) |
| `frontier_llm` | 경쟁 AI model 개발에 도움이 될 수 있는 요청 |
| `reasoning_extraction` | model의 [summarized thinking](https://platform.claude.com/docs/en/build-with-claude/extended-thinking#summarized-thinking)을 추출하려는 요청 |

Classifier 차단은 **model refusal**(model 자체가 다른 정책상의 이유로 거부하는 것)과 구별됩니다. 둘 다 `stop_reason: "refusal"`로 나타나지만, `stop_details.category`를 통해 어떤 classifier가 차단했는지 알 수 있습니다.

> **참고:** Server-side fallback 기능을 사용하지 **않을** 때 `stop_details`에는 `fallback_credit_token`도 포함됩니다. 이를 사용하면 fallback model 요청을 cache read로 과금할 수 있습니다. [과금 변경 사항](#4-billing-changes)을 참고하세요.


## 2. Server-side fallback(권장)

Messages API가 fallback을 대신 실행할 수 있습니다. `[{"model": "claude-opus-4-8"}]`을 담은 `fallbacks`와 `server-side-fallback-2026-06-01` beta header를 전달하세요. Fable의 classifier가 turn을 차단하면 API가 Opus 4.8로 자동 재시도하며, 무슨 일이 있었는지 알 수 있도록 표시합니다.

자동 fallback 기능은 현재 **Claude API**와 **Claude Platform on AWS**에서 지원됩니다. 현재는 Fable 5에서 Opus 4.8로의 fallback만 지원하며, 지원 범위를 확대할 예정입니다.

```bash
curl https://api.anthropic.com/v1/messages \
  -H "x-api-key: $ANTHROPIC_API_KEY" \
  -H "anthropic-version: 2023-06-01" \
  -H "anthropic-beta: server-side-fallback-2026-06-01" \
  -H "content-type: application/json" \
  -d '{
    "model": "claude-fable-5",
    "max_tokens": 1024,
    "fallbacks": [
      { "model": "claude-opus-4-8" }
    ],
    "messages": [
      { "role": "user", "content": "Hello, world" }
    ]
  }'
```

### Fallback을 실행할 수 없는 경우

`fallbacks`를 구성했지만 rate limit 소진이나 과부하로 인해 API가 fallback model에 접근할 수 없는 경우, turn은 여전히 refusal로 반환되며 `stop_details.recommended_model`에는 직접 재시도할 canonical model id가 지정됩니다:

```json
{
  "stop_reason": "refusal",
  "stop_details": {
    "type": "refusal",
    "category": "cyber",
    "recommended_model": "claude-opus-4-8"
  }
}
```

`recommended_model`은 이 경우(`fallbacks`를 구성했고 *동시에* fallback을 실행할 수 없었던 경우)에**만** 채워집니다. Fallback을 구성하지 않은 일반적인 차단에는 이 필드가 존재하지 않습니다. 이것이 [1절](#1-what-a-classifier-block-looks-like)의 기본 예제에 이 필드가 없는 이유입니다.


In [ ]:
# Fable은 추가 safety filter를 적용합니다. Fallback chain을 구성하면 API가
# server-side에서 차단된 turn을 다음 model로 재시도합니다. stop_reason이 "refusal"이면
# chain 전체가 거부했다는 의미입니다.

from anthropic import Anthropic

client = Anthropic()


def chat_turn(messages, max_tokens=1024):
    """API를 한 번 호출하며, server가 fallback을 처리합니다."""
    return client.beta.messages.create(
        model=PRIMARY_MODEL,
        max_tokens=max_tokens,
        messages=messages,
        betas=[SERVER_SIDE_FALLBACK_BETA],
        fallbacks=[{"model": FALLBACK_MODEL}],
    )


### Fallback 감지(non-streaming)

Fallback 응답은 각 전환 지점마다 `{"type": "fallback"}` content block을 포함하며, `usage.iterations`는 model별 사용량을 기록합니다. 대화의 이전 turn에서 fallback이 발생하여 fallback model로 바로 전달되는 **sticky-served** turn에는 fallback block이 *없습니다*. 요청이 직접 전달되어 `content`에 표시할 경계가 없기 때문입니다. `usage.iterations`는 fallback model이 해당 turn을 처리했는지 확인하는 신뢰할 수 있는 방법입니다.


In [ ]:
def fallback_hops(response):
    """이 turn을 실행하고 차단한 각 hop의 (from_model, to_model)."""
    hops = []
    for b in response.content:
        if getattr(b, "type", None) == "fallback":
            d = b.model_dump() if hasattr(b, "model_dump") else dict(b)
            hops.append((d["from"]["model"], d["to"]["model"]))
    return hops


def served_by_fallback(response):
    """Fallback model이 응답을 처리했으면 True이며, fallback block이 없는
    sticky-served turn도 포함합니다. usage.iterations는 fallback model이
    turn을 처리했는지 확인하는 가장 좋은 방법입니다."""
    iters = getattr(response.usage, "iterations", None) or []
    return any(
        (i.get("type") if isinstance(i, dict) else getattr(i, "type", None))
        == "fallback_message"
        for i in iters
    )


response = chat_turn([{"role": "user", "content": "Hello, world"}])
hops = fallback_hops(response)
for from_model, to_model in hops:
    print(f"[{from_model} blocked \u2014 continued on {to_model}]")
if not hops and served_by_fallback(response):
    print(f"[sticky: served directly by {response.model}]")


### Streaming 중 fallback 감지

Block이 `{"type": "fallback"}`인 `content_block_start` event를 확인하세요. 이는 stream 내 전환 지점을 나타냅니다. 하지만 turn별 확정적인 판단을 위해서는 non-streaming 응답과 마찬가지로 **최종** message의 `usage.iterations`를 확인하세요. Fallback model이 직접 처리한 stream을 포함하여 모든 경우에 이 검사가 신뢰할 수 있으므로, stream 내 event에만 의존하지 말고 이를 판단의 기준으로 사용하세요.


In [ ]:
with client.beta.messages.stream(
    model=PRIMARY_MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hello, world"}],
    betas=[SERVER_SIDE_FALLBACK_BETA],
    fallbacks=[{"model": FALLBACK_MODEL}],
) as stream:
    for event in stream:
        if (
            getattr(event, "type", None) == "content_block_start"
            and getattr(event.content_block, "type", None) == "fallback"
        ):
            fb = event.content_block
            fb = fb.model_dump() if hasattr(fb, "model_dump") else dict(fb)
            print(f"[switching: {fb['from']['model']} -> {fb['to']['model']}]")
    final = stream.get_final_message()

# Non-streaming과 동일한 확정적인 turn별 검사: usage.iterations는 fallback model이
# 직접 처리한 stream(stream 내 event 없음)도 감지합니다.
if served_by_fallback(final):
    print(f"[fallback model served this stream: {final.model}]")


### Fallback 응답의 구조

Fallback 응답에는 `message.model`(최종적으로 응답한 model), 각 전환 지점을 표시하는 `{"type": "fallback"}` content block, 그리고 `usage.iterations`의 시도별 사용량이 포함됩니다:

```json
{
  "id": "msg_01Ab...",
  "type": "message",
  "role": "assistant",
  "model": "claude-opus-4-8",
  "content": [
    { "type": "fallback", "from": { "model": "claude-fable-5" }, "to": { "model": "claude-opus-4-8" } },
    { "type": "text", "text": "..." }
  ],
  "stop_reason": "end_turn",
  "stop_details": null,
  "usage": {
    "input_tokens": 412, "output_tokens": 264,
    "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0,
    "iterations": [
      { "type": "message", "model": "claude-fable-5", "input_tokens": 408, "output_tokens": 0,
        "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0 },
      { "type": "fallback_message", "model": "claude-opus-4-8", "input_tokens": 412, "output_tokens": 264,
        "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0 }
    ]
  }
}
```

**시도별 override.** 각 fallback 항목은 해당 시도에 한해서 `max_tokens`, `thinking`, `output_config`, `speed`를 override할 수 있습니다(`output_config`와 `speed`에는 해당 top-level field와 동일한 beta header도 필요합니다). 항목의 override를 병합한 요청은 해당 항목의 model에 대한 올바른 형식의 직접 요청이어야 합니다.

```json
{
  "model": "claude-fable-5",
  "max_tokens": 1024,
  "fallbacks": [
    { "model": "claude-opus-4-8", "max_tokens": 8192, "thinking": {"type": "disabled"}, "speed": "fast" }
  ],
  "messages": [
    { "role": "user", "content": "Hello, world" }
  ]
}
```

**과금.** `usage.input_tokens`는 turn당 한 번 계산됩니다. `usage.output_tokens`는 응답을 반영합니다. 정확한 model별 내역이 필요하면 `usage.iterations`를 사용하세요.


## 3. Streaming

Streaming에서 fallback은 자동으로 작동하도록 설계되었습니다. Classifier가 **출력이 사용자에게 도달하기 전에** 차단하면, stream은 fallback model의 응답으로 시작합니다. 이 재시도는 외부에 드러나지 않으며 fallback SSE event가 발생하지 않습니다.

Classifier가 **stream 중간에** 차단하는 경우에도 같은 stream에서 재시도됩니다. 부분 출력은 유지되고, `{"type": "fallback"}` content block이 경계를 표시하며, fallback model이 부분 출력에 이어서 응답합니다. 이미 streaming된 내용은 절대 버려지지 않습니다.


In [ ]:
def stream_turn(messages, max_tokens=1024):
    with client.beta.messages.stream(
        model=PRIMARY_MODEL,
        max_tokens=max_tokens,
        messages=messages,
        betas=[SERVER_SIDE_FALLBACK_BETA],
        fallbacks=[{"model": FALLBACK_MODEL}],
    ) as stream:
        # 이미 streaming된 내용은 절대 버려지지 않습니다. Stream 중간 차단 이후
        # 최종 message는 부분 출력 + fallback block + 이어지는 응답입니다.
        final = stream.get_final_message()

    if final.stop_reason == "refusal":
        return final  # chain 전체가 거부함
    text = "".join(b.text for b in final.content if b.type == "text")
    print(f"{final.model}: {text}")
    return final


## 4. 과금 변경 사항

이 절에서는 차단된 요청과 fallback에 요금이 부과되는 방식을 다룹니다. Fallback과 Anthropic SDK helper를 사용하면 이러한 동작은 자동으로 적용됩니다. **Cache-read 과금 변경을 적용할 때에만 조치가 필요하며, 그중에서도 server-side fallback을 사용하지 않는 경우에만 필요합니다.**

**1. 일부 category에서는 streaming 시작 전 차단에도 요금이 부과됩니다.** Output token이 반환되기 전에 차단된 요청의 `stop_details.category`가 `"bio"`, `"frontier_llm"`, `"reasoning_extraction"`이면, 다른 요청과 마찬가지로 요청한 model의 요율에 따라 과금됩니다. 2026년 9월 23일 기준으로, 그 외 category이거나 category가 없는 streaming 시작 전 차단에는 요금이 부과되지 않습니다. 현재 목록은 [Refusals and fallback](https://platform.claude.com/docs/en/build-with-claude/refusals-and-fallback#how-refusals-are-billed)에 있습니다. 과금은 safeguard의 false-positive 비율이 낮은 category에만 적용되며, 주로 safeguard를 우회하려는 반복적인 시도의 비용을 높입니다. 과금되는 차단 이후 fallback하면 fallback 요청에도 과금되어 두 요청 모두 요금이 부과되지만, 아래 2번 항목에 따라 재시도의 prompt-cache 비용이 상쇄됩니다.

Stream 중간(output token의 streaming이 시작된 이후)에 차단된 요청은 category와 관계없이 input token과 이미 streaming된 output에 대해 일반 요율로 과금됩니다.

**2. Fable 5 → Opus 4.8 fallback의 input token은 cache hit로 과금됩니다.** 보통 다른 model로 전환하면 [cache *write*](https://platform.claude.com/docs/en/build-with-claude/prompt-caching#how-prompt-caching-works)로 과금되며, 이는 기본 input-token 비용의 1.25배(5분 TTL) 또는 2배(60분 TTL)입니다. 대신 여기서는 해당 Opus token이 이미 cache되어 있는 것처럼, 즉 기본 input-token 가격의 10%인 cache *read*로 과금합니다.

- **Server-side fallback 기능을 사용하는 경우:** 이 과금 변경은 자동으로 적용됩니다.
- **Server-side fallback 기능을 사용하지 않는 경우:** 아래 credit-token 흐름을 참고하세요.

### Fallback credit token 사용(client-side fallback만 해당)

Safety classifier에 의해 차단된 Fable 요청은 `stop_details`에 `fallback_credit_token`을 포함합니다. Token은 차단된 요청에 과금 대상인 cached prefix가 있었던 경우에**만** 존재하며, 그 외에는 `null`입니다.

사용 방법:

1. 이후 Opus 4.8 요청을 보낼 때 `anthropic-beta: fallback-credit-2026-06-01` header를 포함하세요.
2. Token을 top-level `fallback_credit_token` parameter로 전달하세요.
3. Prompt를 구성하는 field를 차단된 요청과 **동일하게** 유지하세요. `system`, `messages`, `tools`가 정확히 같아야 합니다.

그러면 Fable 요청에서 cache된 prefix는 cache-write 요율 대신 cache-read 요율로 과금됩니다. 전환 비용이 환불되어, 재시도 비용은 처음부터 Opus로 대화했을 때와 같아집니다.

**유효 조건:** Token은 차단된 Fable 5 요청으로부터 **5분 이내**에 발생하며 **동일한 org 및 workspace**에서 시작된 Opus 4.8 요청에서만 유효합니다.

**Stream 중간 차단.** Fable 5 요청이 output token을 streaming하는 도중 차단되면, `stop_details`에는 credit token과 함께 `fallback_has_prefill_claim: true`도 포함됩니다. 이는 이후 Opus 4.8 요청에서 해당 부분 출력을 assistant prefill로 추가하여 Fable이 멈춘 지점부터 이어갈 수 있다는 의미입니다. Opus 4.8 요청에서는 [일반적으로 허용되지 않는](https://platform.claude.com/docs/en/test-and-evaluate/strengthen-guardrails/increase-consistency#prefill-claudes-response) 기능입니다.


In [ ]:
def redeem_credit_after_block(blocked_response, messages, system=None, tools=None, max_tokens=1024):
    """Classifier에 의해 차단된 Fable turn을 Opus 4.8로 재시도하면서
    fallback credit token을 사용하여 cached prefix가 cache-read 요율로
    과금되게 합니다. Server-side fallback을 사용하지 않을 때만 사용하세요.

    차단된 요청과 동일한 `system`, `messages`, `tools`를 전달하세요."""
    details = blocked_response.stop_details
    credit = getattr(details, "fallback_credit_token", None) if details else None

    # system, messages, tools는 차단된 요청과 정확히 동일해야 합니다.
    kwargs = {"model": FALLBACK_MODEL, "max_tokens": max_tokens, "messages": messages}
    if system is not None:
        kwargs["system"] = system
    if tools is not None:
        kwargs["tools"] = tools
    if credit is not None:  # 차단된 요청에 cached prefix가 있었을 때만 존재함
        kwargs["betas"] = [FALLBACK_CREDIT_BETA]
        kwargs["extra_body"] = {"fallback_credit_token": credit}

    return client.beta.messages.create(**kwargs)


## 5. SDK를 사용한 client-side fallback

Server-side fallback은 native Claude API와 Claude Platform on AWS에서 사용할 수 있지만, 현재 Amazon Bedrock, Vertex AI, Microsoft Foundry, Message Batches API에서는 사용할 수 없습니다. 이들 플랫폼을 사용하는 경우나 fallback 로직을 client에 두고 싶은 경우를 위해, Anthropic SDK(Python, TypeScript, Go, Java, C#)는 **refusal-fallback middleware**를 제공합니다.

Fallback model 목록과 `BetaFallbackState`를 사용해 client에 한 번 구성한 뒤, 평소처럼 `client.beta.messages`를 호출하세요. Middleware는 다음을 수행합니다:

- `stop_reason: "refusal"` turn을 목록의 다음 model로 재시도합니다(fallback도 거부하면 chain을 계속 따라가며, 모든 항목이 거부하면 예외를 발생시키는 대신 원래 refusal을 반환합니다).
- **모든 요청에 `fallback-credit-2026-06-01` beta header를 자동으로 전송**하므로, 직접 token을 관리하지 않고도 [4절](#4-billing-changes)의 cache-read 과금 변경을 적용받습니다.
- 대화 기록의 `fallback` content block을 대신 관리합니다.
- 요청을 수락한 model을 `BetaFallbackState`에 기록하여 이후 turn도 해당 model로 고정합니다.

이는 server-side `fallbacks` parameter와 **상호 배타적**입니다. 둘 중 하나를 사용하세요. (Middleware를 설치한 app에서 server-side `fallbacks` 요청을 보내려면, middleware가 없는 별도의 client instance를 사용하세요.)


In [ ]:
from anthropic import Anthropic, BetaFallbackState, BetaRefusalFallbackMiddleware

# Fallback chain으로 middleware를 한 번 설치합니다. 요청별 beta는 필요하지 않습니다.
client = Anthropic(
    middleware=[BetaRefusalFallbackMiddleware([{"model": FALLBACK_MODEL}])],
)

state = BetaFallbackState()  # 이후 turn을 요청을 수락한 model에 고정하도록 turn 간에 재사용함

# Non-streaming: 거부된 Fable turn을 별도 처리 없이 Opus 4.8로 재시도합니다.
with state:
    message = client.beta.messages.create(
        model=PRIMARY_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": "Hello, Claude"}],
    )
print(f"served by: {message.model}")

# Streaming: refusal 시 middleware가 fallback model의 event를
# 열려 있는 같은 stream에 이어 붙입니다.
with (
    state,
    client.beta.messages.stream(
        model=PRIMARY_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": "Hello, Claude"}],
    ) as stream,
):
    for event in stream:
        if event.type == "text":
            print(event.text, end="", flush=True)
    final = stream.get_final_message()
print(f"\nserved by: {final.model}")


## 6. 흔한 anti-pattern

**계정당 한 번이 아니라 모든 요청에 fallback을 설정하세요.** Opus 4.8 fallback을 활성화하는 계정 수준이나 session 수준의 switch는 없습니다. 각 API 호출에 fallback 구성이 포함되어야 합니다. Fallback을 켜지 않은 호출은 fallback model로 조용히 재시도하는 대신 refusal을 반환합니다.

**요청을 만드는 모든 코드 경로를 점검하세요.** 재시도 버튼, message 재생성, tool-use continuation 같은 기능은 자체적으로 요청을 구성하는 경우가 많으며, 각 요청에서 fallback 구성이 눈에 띄지 않게 빠질 수 있습니다. 모든 진입점에 fallback을 명시적으로 설정하세요. (Codebase 전반에 fallback을 빠르게 추가하려면 [migration guide](https://platform.claude.com/docs/en/about-claude/models/migration-guide)와 [claude-api skill](https://platform.claude.com/docs/en/agents-and-tools/agent-skills/claude-api-skill)을 참고하세요.)

**Subagent 호출에도 fallback을 포함하고 agent별 동작을 예상하세요.** Application이 하나의 session에서 여러 agent를 실행한다면 모든 agent의 호출에 fallback 구성이 필요합니다. Refusal이 발생하면 이를 받은 agent만 fallback model로 이동하며, 다른 agent는 Fable을 계속 사용합니다. 한 agent의 fallback이 session 전체에 적용된다고 가정하지 마세요. (Claude Code의 sub-agent도 마찬가지입니다. Refusal을 만난 sub-agent만 Opus로 fallback하고, 나머지 session은 Fable 5를 계속 사용합니다.)

**Refusal 이후 같은 요청을 다시 제출하면 다시 거부될 뿐입니다.** 거부된 콘텐츠가 여전히 대화 기록에 있으므로, 같은 model에 다시 제출하면 차단이 다시 발생합니다. `stop_reason: "refusal"`일 때는 fallback model로 재시도하고, 대화의 나머지 부분도 fallback model에 유지해야 함을 router가 알 수 있도록 별도의 표시를 설정하세요.

**요청한 model이 아니라 `usage.iterations`를 기준으로 실제 응답 model의 분석 지표를 구성하세요.** 응답의 `model` field는 실제로 응답한 model이므로, fallback으로 처리된 turn에는 Opus 4.8이 표시됩니다. *요청한* model을 기준으로 기록한 분석 지표는 fallback을 사용할 때마다 잘못됩니다. 신뢰할 수 있는 turn별 검사는 최종 usage 기록의 `usage.iterations`입니다.

**Streaming 잘림을 주의해서 처리하세요.** Stream 중간에 classifier 차단을 만나면 fallback 앞에 나타나는 `thinking`, `tool_use` 또는 기타 block은 제외하세요. 잘린 `tool_use` block은 parsing할 수 없는 JSON이며, 다른 model의 `thinking` block은 다음 호출을 실패하게 합니다. Server-side fallback 없이 계속 진행한다면, 부분 응답을 완료된 assistant turn에 붙여 넣는 대신 `stop_details`의 `fallback_has_prefill_claim` 허용을 사용하세요.
